Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Agent patterns

- A supervisor decides which worker runs next, and when the job is finished
- The workers know nothing about each other ; all routing goes through it

One supervisor over two workers.

### Exercise
Agent Supervisor
1. Change the question assigned to the topic variable and run the code again.
2. Change the prompts for the "researcher" and "expert" nodes so that they take on different roles, e.g. "screenwriter" and "set designer", and run the code again.

In [1]:
import operator
from typing import Annotated, List, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv
load_dotenv()

llm = make_llm()

class SupervisorState(TypedDict):
    """State for supervisor pattern with multiple agents."""
    topic: str
    messages: Annotated[List[str], operator.add]
    next_agent: str
    final_answer: str


def researcher_agent(state: SupervisorState) -> dict:
    """Researcher agent gathers information about the topic."""
    sys = ( # Modify the prompt below
        "You are a researcher. Your job is to gather key facts and information "
        "about the given topic. Provide 2-3 key points. Be concise."
    )
    messages_for_llm = [
        ("system", sys),
        ("user", f"Research this topic: {state['topic']}")
    ]
    resp = llm.invoke(messages_for_llm)
    research_msg = f"RESEARCHER: {resp.content}"
    return {"messages": [research_msg]}


def expert_agent(state: SupervisorState) -> dict:
    """Expert agent analyzes and provides insights based on research."""
    sys = ( # Modify the prompt below
        "You are an expert analyst. Review the research provided and give "
        "your expert analysis and conclusions. Be specific and insightful."
    )
    # Get context from previous messages
    context = "\n".join(state["messages"])
    messages_for_llm = [
        ("system", sys),
        ("user", f"Topic: {state['topic']}\n\nPrevious research:\n{context}\n\nProvide your expert analysis.")
    ]
    resp = llm.invoke(messages_for_llm)
    expert_msg = f"EXPERT: {resp.content}"
    return {"messages": [expert_msg]}


def supervisor_agent(state: SupervisorState) -> dict:
    """Supervisor decides which agent should act next or if discussion should end."""
    sys = (
        "You are a supervisor managing a research discussion between a RESEARCHER and an EXPERT. "
        "Based on the conversation so far, decide what should happen next:\n"
        "- Return 'researcher' if we need initial research or more information\n"
        "- Return 'expert' if research is done and we need expert analysis\n"
        "- Return 'end' if both research and expert analysis are complete\n\n"
        "Respond with ONLY one word: researcher, expert, or end"
    )

    context = "\n".join(state["messages"]) if state["messages"] else "No discussion yet"
    messages_for_llm = [
        ("system", sys),
        ("user", f"Topic: {state['topic']}\n\nConversation:\n{context}\n\nWhat's next?")
    ]
    resp = llm.invoke(messages_for_llm)
    next_step = resp.content.strip().lower()

    # Ensure valid response
    if next_step not in ["researcher", "expert", "end"]:
        next_step = "end"

    return {"next_agent": next_step}


def finalize_answer(state: SupervisorState) -> dict:
    """Compile final answer from the discussion."""
    sys = (
        "Summarize the research discussion into a clear, concise final answer. "
        "Include key findings and expert insights."
    )
    context = "\n".join(state["messages"])
    messages_for_llm = [
        ("system", sys),
        ("user", f"Topic: {state['topic']}\n\nDiscussion:\n{context}\n\nProvide final summary:")
    ]
    resp = llm.invoke(messages_for_llm)
    return {"final_answer": resp.content}


def route_supervisor(state: SupervisorState) -> str:
    """Route based on supervisor's decision."""
    next_agent = state.get("next_agent", "researcher")
    if next_agent == "end":
        return "finalize"
    return next_agent

supervisor_graph = StateGraph(SupervisorState)

supervisor_graph.add_node("supervisor", supervisor_agent)
supervisor_graph.add_node("researcher", researcher_agent)
supervisor_graph.add_node("expert", expert_agent)
supervisor_graph.add_node("finalize", finalize_answer)

supervisor_graph.add_edge(START, "supervisor")

supervisor_graph.add_conditional_edges(
    "supervisor",
    route_supervisor,
    {
        "researcher": "researcher",
        "expert": "expert",
        "finalize": "finalize"
    }
)

supervisor_graph.add_edge("researcher", "supervisor")
supervisor_graph.add_edge("expert", "supervisor")

supervisor_graph.add_edge("finalize", END)

supervisor_agent_graph = supervisor_graph.compile(debug=True)

topic = "What are the main benefits of using LangGraph for building AI agents?" # <- modify the given question

initial_state = {
    "topic": topic,
    "messages": [],
    "next_agent": "",
    "final_answer": ""
}

result = supervisor_agent_graph.invoke(initial_state)

print(f"TOPIC: {topic}\n")
print("=" * 80)
print("\nDISCUSSION:")
print("-" * 80)
for msg in result["messages"]:
    print(f"\n{msg}\n")
print("=" * 80)
print(f"\nFINAL ANSWER:\n{result['final_answer']}")

[values] {'topic': 'What are the main benefits of using LangGraph for building AI agents?', 'messages': [], 'next_agent': '', 'final_answer': ''}
[updates] {'supervisor': {'next_agent': 'researcher'}}
[values] {'topic': 'What are the main benefits of using LangGraph for building AI agents?', 'messages': [], 'next_agent': 'researcher', 'final_answer': ''}
[updates] {'researcher': {'messages': ['RESEARCHER: 1. **Modular Design**: LangGraph offers a modular architecture that allows developers to easily integrate various components and functionalities, facilitating the rapid development and deployment of AI agents tailored to specific tasks.\n\n2. **Enhanced Natural Language Processing**: The platform leverages advanced natural language processing capabilities, enabling AI agents to understand and generate human-like text, which improves user interaction and engagement.\n\n3. **Scalability and Flexibility**: LangGraph is designed to scale efficiently, accommodating a wide range of applicat

### Try a third worker

- Add one and mention it in the supervisor's instructions
- Then take it back out of the instructions but leave it in the graph. A
  worker the supervisor was never told about is never chosen